In [ ]:
import re
import fitz  # PyMuPDF
import pandas as pd
import streamlit as st
import io

def extract_text_from_pdf(pdf_file):
    """Trích xuất toàn bộ text từ file PDF (file object từ Streamlit)."""
    text = ""
    with fitz.open(stream=pdf_file.read(), filetype="pdf") as pdf:
        for page in pdf:
            text += page.get_text("text")
    return text

with open(r"E:\D\HOCDIBANTRE\AnhSa\20251019_SGN_CHNLASC0483158_JP_PARTA_1.pdf", "rb") as f:
    text = extract_text_from_pdf(f)

In [67]:
import pdfplumber
import fitz

# fitz -> lấy vị trí và layout
doc = fitz.open(r"E:\D\HOCDIBANTRE\AnhSa\20251019_SGN_CHNLASC0483158_JP_PARTA_1.pdf")

# pdfplumber -> lấy bảng, text có cấu trúc
with pdfplumber.open(r"E:\D\HOCDIBANTRE\AnhSa\20251019_SGN_CHNLASC0483158_JP_PARTA_1.pdf") as pdf:
    text = "\n".join(page.extract_text() or "" for page in pdf.pages)


In [106]:
import re
import pandas as pd

def find_text_before_keyword(text, keyword):
    """
    Extract the number (or text) immediately before the given keyword.
    Example: '31 Cartons of Footwear Division Goods' -> '31'
    """
    pattern = rf'(\d+)\s+(?={re.escape(keyword)})'
    match = re.search(pattern, text)
    return match.group(1) if match else None

def find_text_after_keyword(text, keyword, num_chars=50):
    """
    Tìm đoạn văn bản sau một keyword. 
    Nếu cùng dòng không có nội dung, lấy nội dung ở dòng kế tiếp.
    """
    # 1️⃣ Tìm trên cùng dòng
    pattern_same_line = re.escape(keyword) + r"[ \t]*(.{1," + str(num_chars) + r"})"
    match = re.search(pattern_same_line, text)
    if match:
        result = match.group(1).strip()
        if result:
            return result

    # 2️⃣ Nếu không có, tìm ở dòng kế tiếp
    pattern_next_line = re.escape(keyword) + r"\s*\n\s*(.{1," + str(num_chars) + r"})"
    match = re.search(pattern_next_line, text)
    if match:
        return match.group(1).strip()

    return None

def split_by_bill_names(text, bill_names):
    """
    Tách text thành các phần tương ứng với bill_names (theo thứ tự trong danh sách).
    Mỗi bill_name chỉ xuất hiện 1 lần, lấy nội dung từ bill_i đến bill_(i+1).
    """
    sections = []
    text_lower = text.lower()
    positions = []
    for b in bill_names:
        # print("Checking bill name:", b)
        if b == "WAYBILL":
            idx = text_lower.find(b.lower())
            positions.append((idx, b))
        else:
        # Regex: tìm tất cả vị trí 'Invoice 1' KHÔNG có '(continue)' ngay sau
            for match in re.finditer(rf'\b{re.escape(b)}\b(?!\s*\(continued\))', text_lower, re.IGNORECASE):
                # print(f"Found '{b}' at position {match.start()}")
                positions.append((match.start(), b))

    
    positions.sort(key=lambda x: x[0])
    
    # Tách nội dung giữa các bill
    for i, (start_idx, bill) in enumerate(positions):
        end_idx = positions[i+1][0] if i + 1 < len(positions) else len(text)
        content = text[start_idx + len(bill): end_idx].strip()
        sections.append((bill, content))
    return sections

def extract_bill_sections(text, bill_names, keyword_dict, verbose=False):
    """
    Extract info for each bill section based on grouped keywords.
    Each main key in keyword_dict will become a column name.
    """
    sections = split_by_bill_names(text, bill_names)
    results = []

    for bill_name, content in sections:
        if verbose:
            print(f"Processing section: {bill_name}")
            print(f"Snippet: {content}...")
            print("-" * 50)

        
        entry = {"Bill Name": bill_name}
        for key, kw_list in keyword_dict.items():
            if not kw_list:
                entry[key] = "⚠️ Not Found"
                continue
            idx = bill_names.index(bill_name)
            kw = kw_list[idx]
            if kw == "":
                entry[key] = None
                continue
            if kw == "Cartons of Footwear Division Goods" or kw == "Cartons of Footwear Division of goods":
                cont = find_text_before_keyword(content, kw)   
            else:           
                cont = find_text_after_keyword(content, kw)

            if cont:
                entry[key] = cont
            else:
                entry[key] = None

        results.append(entry)

    return pd.DataFrame(results)

bill_names = [
    "WAYBILL",
    "Trading Company Commercial Invoice",
    "Factory Commercial Invoice",
    "Factory Packing List",
    "MULTIPLE COUNTRY OF ORIGIN DECLARATION",
    "Japan Customs Form"
]

keywords = {
    "INV": ["Invoice#:", "Reference Invoice #:", "Invoice Number:", "Invoice Number.:", "INVOICE NO.", ""],
    "Total weight": ["", "Total Gross Weight:", "Total Gross Weight:", "Total Gross Kgs:", "",""],
    "PO": ["PO-Item:", "PO#:", "Reference PO#:", "Reference PO#:", "P.O. #:", ""],
    "PO line": ["", "PO Line Item Seq.#: ", "PO Line Item Seq. #:", "Item Seq.:", "ITEM :", ""],
    "Style": ["Material:", "Material#:", "Material #:", "Material:", "MATERIAL", ""],
    "total carton": ["Cartons of Footwear Division of goods", "Cartons of Footwear Division Goods",  "Cartons of Footwear Division Goods", "", "", ""],
    "total quantity": ["Qty:", "Total Invoice ", "Total Invoice Quantity:", "", "",""]
}

df = extract_bill_sections(text, bill_names, keywords, verbose=True)
for i in range(len(df)):
    if "-" in str(df.loc[i, "PO"]):
        parts = df.loc[i, "PO"].split("-")
        df.loc[i, "PO"] = parts[0]
        df.loc[i, "PO line"] = parts[1] if len(parts) > 1 else None
    df["PO"] = df["PO"].astype(str).str.split(",").str[0]
    df["PO"] = df["PO"].astype(str).str.split(" ").str[0]
    df["PO line"] = df["PO line"].astype(str).str.split(",").str[0]
    df["Style"] = df["Style"].astype(str).str.split(",").str[0]
    df["Style"] = df["Style"].astype(str).str.split(" ").str[0]
    df["PO line"] = df["PO line"].astype(str).str.split(" ").str[0]
    df["INV"] = df["INV"].astype(str).str.split(" ").str[0]
    df["total quantity"] = df["total quantity"].astype(str).str.split(" ").str[0]
    df["total carton"] = df["total carton"].astype(str).str.split("  ").str[0]
    df["total volume"] = None
df


Processing section: WAYBILL
Snippet: LE THANH TON STREET, SAI GON WARD, WAYBILL NUMBER
HO CHI MINH CITY, VIETNAM NON NEGOTIABLE
ASC0483158
TAX# 0312658789 ON BEHALF OF (*)
CONSIGNEE EXPORT REFERENCES
NIKE JAPAN GROUP LLC
(TAX ID: 1010703002010)
MIDTOWN TOWER, 9-7-1 AKASAKA,
MINATO-KU, TOKYO 107-6210,
JAPAN (**)
NOTIFY PARTY, Carrier not to be responsible for failure to notify
KINTETSU WORLD EXPRESS ,INC.
(ON BEHALF OF NIKE JAPAN) CARRIER: CMA CGM Asia Shipping Pte. Ltd.
TOKYO SEA IMPORT CLEARANCE CENTER Head Office: #15-01 The Metropolis, Tower 1
6F SUMITOMO FUDOSAN SHIBA BLDG 9 North Buona Vista Drive, Singapore 138588
Tel: (65) 6278 9000 - Fax: (65) 6278 4900
NO 3 1-7-17 SHIBA MINATO-KU
TOKYO,105-0014, JAPAN >
PRE CARRIAGE BY* PLACE OF RECEIPT* FREIGHT TO BE PAID AT NUMBER OF ORIGINAL WAYBILLS
TOKYO ZERO (0)
VESSEL PORT OF LOADING PORT OF DISCHARGE FINAL PLACE OF DELIVERY*
PANAY HO CHI MINH CITY TOKYO
MARKS AND NOS NO AND KIND DESCRIPTION OF PACKAGES AND GOODS AS STATED BY SHIPPER GR

,Bill Name,INV,Total weight,PO,PO line,Style,total carton,total quantity,total volume
0,WAYBILL,PCVJ2509981,None,3503842282,90,HV9981-200,31,68,None
1,Trading Company Commercial Invoice,PCVJ2509981,52.49 Kgs,3503842282,00090,HV9981-200,31,68,None
2,Factory Commercial Invoice,PCVJ2509981,52.49Kgs,3503842282,00090,HV9981-200,31,68,None
3,Factory Packing List,PCVJ2509981,52.49,3503842282,00090,HV9981-200,None,None,None
4,MULTIPLE COUNTRY OF ORIGIN DECLARATION,PCVJ2509981,None,3503842282,90,HV9981-200,None,None,None
5,Trading Company Commercial Invoice,PCVJ2509987,49.11 Kgs,3503842282,00100,HV9981-600,27,64,None
6,Factory Commercial Invoice,PCVJ2509987,49.11Kgs,3503842282,00100,HV9981-600,27,64,None
7,Factory Packing List,PCVJ2509987,49.11,3503842282,00100,HV9981-600,None,None,None
8,MULTIPLE COUNTRY OF ORIGIN DECLARATION,PCVJ2509987,None,3503842282,100,HV9981-600,None,None,None
9,Trading Company Commercial Invoice,PCVJ2508881,78.53 Kgs,3503830020,00020,IM8053-237,38,100,None


In [40]:
keywords = {
    "INV": ["Invoice#:", "Invoice Number:", "Invoice Number.:", "INVOICE NO."],
    "Total weight": ["Total Gross Kgs:"],
    "total volume": [],
    "PO": ["PO-Item:", "P.O.#:", "PO#:", "PO Number:"],
    "PO line": ["ITEM:", "PO Line No:", "PO Line#:"],
    "Style": ["Material:", "Material No:", "Material#:", "MATERIAL"],
    "total carton": ["Total Carton:", "Total Cartons:", "Total No. of Cartons:"],
    "total quantity": ["Total Quantity:", "Total Qty:", "Qty:"]
}